# QLoRA fine-tuning & evaluation for S3 table extraction (Team 35, RQ2)

**RQ2:** *Does QLoRA adaptation improve structured extraction on complex, multi-layout, and
vector-outline/OCR government PDF tables?*

This notebook is self-contained and runs on a **free Google Colab T4 (16 GB)**:

1. loads `eval/qlora_train.jsonl` (66 examples, text-layer reports 2020-21, 2022-23, 2023-24) and
   `eval/eval_set.json` (held-out S3 cases: 2021-22 OCR pages + 2024-25) and **asserts zero overlap**;
2. evaluates the base model `Qwen/Qwen2.5-1.5B-Instruct` (4-bit) on the held-out cases → **Base F1**;
3. trains a 4-bit **QLoRA** adapter (r = 16, alpha = 32) and logs GPU, time and hyper-parameters;
4. re-evaluates the **same** cases → **Adapter F1**, **Delta F1**, one before/after example,
   saves `eval/adapter_comparison.csv` and zips the adapter for upload.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run all cells in order.
Scoring = field-level precision / recall / F1 over the 5 region values of each case
(the same rule as `eval/run_eval.py`).

## 1. Install libraries

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33"

## 2. Get the data
Option A clones the project branch. If the repository is private, use option B: upload
`qlora_train.jsonl` and `eval_set.json` from the `eval/` folder when asked.

In [ ]:
import os, json
REPO_URL = "https://github.com/AsthaSharma133/Puducherry_Agriculture-_System.git"
BRANCH = "complete-phase3"
DATA_DIR = "Puducherry_Agriculture-_System/puducherry_agri_agent/eval"

if not os.path.exists(DATA_DIR):
    os.system(f"git clone -q -b {BRANCH} {REPO_URL}")
if not os.path.exists(os.path.join(DATA_DIR, "eval_set.json")):
    from google.colab import files          # option B: manual upload
    os.makedirs(DATA_DIR, exist_ok=True)
    uploaded = files.upload()                # choose qlora_train.jsonl and eval_set.json
    for name, content in uploaded.items():
        open(os.path.join(DATA_DIR, name), "wb").write(content)
print(os.listdir(DATA_DIR))

## 3. Load train / test and assert zero overlap

In [ ]:
REGIONS = ["Puducherry", "Karaikal", "Mahe", "Yanam", "UT"]

train = [json.loads(line) for line in open(os.path.join(DATA_DIR, "qlora_train.jsonl"), encoding="utf-8")]
all_cases = json.load(open(os.path.join(DATA_DIR, "eval_set.json"), encoding="utf-8"))["cases"]
test = [c for c in all_cases if c["agent"] == "A2"]          # the held-out S3 PDF cases

def cell_keys_train(rows):
    # (source file, page, crop, region, year) for every value a training example teaches
    return {(r["source_file"], r["page"], r["crop"], reg, r["year"]) for r in rows for reg in r["regions"]}

def cell_keys_test(cases):
    return {(c["input_file"], c["page"], c["crop"], reg, c["year"]) for c in cases for reg in c["expected"]}

shared = cell_keys_train(train) & cell_keys_test(test)
assert not shared, f"train/test overlap: {sorted(shared)[:3]}"
assert not ({r["year"] for r in train} & {c["year"] for c in test}), "a year is used for both"
print(f"train examples: {len(train)}  test cases: {len(test)} "
      f"({sum(c['expected_tool'] == 'ocr_tesseract' for c in test)} OCR)  overlap: 0")

## 4. GPU check and base model (4-bit)

In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print("GPU:", GPU_NAME, GPU_MEM_GB, "GB")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.config.use_cache = True

## 5. Prompt, JSON parsing and field-level scoring

In [ ]:
import re
SYSTEM = "You extract numbers from Indian government statistical tables. Answer with JSON only."

def prompt_text(instruction, table_text):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": instruction + "\n\nTable:\n" + table_text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_json(text):
    # take the first {...} in the answer; return {} if it is not valid JSON
    match = re.search(r"\{.*?\}", text, re.S)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}

def to_float(value):
    try:
        return float(str(value).replace(",", ""))
    except (TypeError, ValueError):
        return None

def score(expected, predicted):
    # TP = right value; FP = a value given but wrong; FN = expected value not given correctly
    tp = fp = fn = 0
    for region, value in expected.items():
        got = to_float(predicted.get(region))
        if got is not None and abs(got - float(value)) < 1e-6:
            tp += 1
        else:
            fn += 1
            if got is not None:
                fp += 1
    return tp, fp, fn

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return round(p, 4), round(r, 4), round(2 * p * r / (p + r) if p + r else 0.0, 4)

@torch.no_grad()
def evaluate(model, cases, label):
    model.eval()
    results = []
    for case in cases:
        inputs = tokenizer(prompt_text(case["instruction"], case["model_input"]), return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        tp, fp, fn = score(case["expected"], parse_json(answer))
        results.append({"case_id": case["case_id"], "subset": case["expected_tool"], "answer": answer.strip(),
                        "tp": tp, "fp": fp, "fn": fn})
    tp, fp, fn = (sum(r[k] for r in results) for k in ("tp", "fp", "fn"))
    print(label, "P/R/F1 =", prf(tp, fp, fn), f"on {len(cases)} cases, {tp + fn} fields")
    return results

## 6. Base model on the held-out cases → Base F1

In [ ]:
t0 = time.time()
base_results = evaluate(model, test, "BASE")
print("base eval seconds:", round(time.time() - t0, 1))

## 7. QLoRA fine-tuning (r = 16, alpha = 32)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

HYPER = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "epochs": 4, "learning_rate": 2e-4,
         "batch_size": 4, "grad_accum": 2, "max_len": 768,
         "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]}

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=HYPER["r"], lora_alpha=HYPER["lora_alpha"],
                                         lora_dropout=HYPER["lora_dropout"], bias="none",
                                         task_type="CAUSAL_LM", target_modules=HYPER["target_modules"]))
model.print_trainable_parameters()

def encode(example):
    # loss only on the answer tokens: prompt tokens get label -100
    prompt = prompt_text(example["instruction"], example["input"])
    answer = example["output"] + tokenizer.eos_token
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:HYPER["max_len"]]
    labels = ([-100] * len(p_ids) + a_ids)[:HYPER["max_len"]]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_data = [encode(e) for e in train]

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id or tokenizer.eos_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        gap = width - len(b["input_ids"])
        out["input_ids"].append([pad] * gap + b["input_ids"])
        out["attention_mask"].append([0] * gap + b["attention_mask"])
        out["labels"].append([-100] * gap + b["labels"])
    return {k: torch.tensor(v) for k, v in out.items()}

args = TrainingArguments(output_dir="qlora_out", num_train_epochs=HYPER["epochs"],
                         per_device_train_batch_size=HYPER["batch_size"],
                         gradient_accumulation_steps=HYPER["grad_accum"],
                         learning_rate=HYPER["learning_rate"], fp16=True, logging_steps=5,
                         save_strategy="no", report_to=[], seed=35)
trainer = Trainer(model=model, args=args, train_dataset=train_data, data_collator=collate)
t0 = time.time()
train_output = trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 1)
TRAIN_LOG = {"gpu": GPU_NAME, "gpu_memory_gb": GPU_MEM_GB, "base_model": MODEL_ID,
             "train_examples": len(train), "train_seconds": TRAIN_SECONDS,
             "final_train_loss": round(train_output.training_loss, 4),
             "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2), **HYPER}
print(json.dumps(TRAIN_LOG, indent=2))

## 8. Adapter on the same held-out cases → Adapter F1 and Delta F1

In [ ]:
model.config.use_cache = True
adapter_results = evaluate(model, test, "ADAPTER")

def summary_rows(subset):
    rows = []
    for name, results in (("base", base_results), ("adapter", adapter_results)):
        picked = [r for r in results if subset == "all" or r["subset"] == subset]
        tp, fp, fn = (sum(r[k] for r in picked) for k in ("tp", "fp", "fn"))
        rows.append((name, len(picked), tp + fn) + prf(tp, fp, fn))
    return rows

for subset in ("all", "ocr_tesseract", "text_layer"):
    (_, n, fields, bp, br, bf), (_, _, _, ap, ar, af) = summary_rows(subset)
    print(f"{subset:<14} n={n:<3} fields={fields:<3} base F1={bf:.3f}  adapter F1={af:.3f}  delta={af - bf:+.3f}")

## 9. One before-vs-after example

In [ ]:
example = next((i for i, (b, a) in enumerate(zip(base_results, adapter_results)) if a["tp"] > b["tp"]), 0)
case = test[example]
print(case["case_id"], case["year"], "Table", case["table"], "col", case["column"], case["crop"])
print("INPUT (raw OCR/text):\n" + case["model_input"])
print("\nEXPECTED:", json.dumps(case["expected"]))
print("BASE    :", base_results[example]["answer"], f"({base_results[example]['tp']}/5 correct)")
print("ADAPTER :", adapter_results[example]["answer"], f"({adapter_results[example]['tp']}/5 correct)")

## 10. Save `eval/adapter_comparison.csv`, the training log and the zipped adapter

In [ ]:
import csv, shutil
csv_path = os.path.join(DATA_DIR, "adapter_comparison.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["row_type", "case_or_subset", "n_cases", "n_fields", "base_precision", "base_recall", "base_f1",
                "adapter_precision", "adapter_recall", "adapter_f1", "delta_f1"])
    for subset in ("all", "ocr_tesseract", "text_layer"):
        (_, n, fields, bp, br, bf), (_, _, _, ap, ar, af) = summary_rows(subset)
        w.writerow(["summary", subset, n, fields, bp, br, bf, ap, ar, af, round(af - bf, 4)])
    for b, a in zip(base_results, adapter_results):
        bp, br, bf = prf(b["tp"], b["fp"], b["fn"])
        ap, ar, af = prf(a["tp"], a["fp"], a["fn"])
        w.writerow(["case", b["case_id"], 1, b["tp"] + b["fn"], bp, br, bf, ap, ar, af, round(af - bf, 4)])
json.dump(TRAIN_LOG, open(os.path.join(DATA_DIR, "qlora_train_log.json"), "w"), indent=2)

model.save_pretrained("qlora_adapter_team35")
tokenizer.save_pretrained("qlora_adapter_team35")
shutil.make_archive("qlora_adapter_team35", "zip", "qlora_adapter_team35")
print("saved", csv_path, "and qlora_adapter_team35.zip")
try:
    from google.colab import files
    files.download(csv_path)
    files.download(os.path.join(DATA_DIR, "qlora_train_log.json"))
    files.download("qlora_adapter_team35.zip")
except ImportError:
    pass

### After the run
Copy `adapter_comparison.csv` and `qlora_train_log.json` into the project's `eval/` folder and
commit them; upload `qlora_adapter_team35.zip` to the team Google Drive (adapter weights are not
stored in git). Report Base F1, Adapter F1 and Delta F1 exactly as printed in section 8.